# Chapter 10: Consistency and Consensus
*From: Designing Data-Intensive Applications*

---

## TL;DR

- **Linearizability** is a *recency* guarantee: the replicated system behaves as if there is a single copy, and every operation takes effect atomically at some moment between its request and response.
- **CAP** is a narrow theorem; the bigger practical story is **PACELC**: even without partitions, any linearizable system pays a latency tax proportional to network-delay uncertainty (Attiya-Welch lower bound).
- **Logical clocks** (Lamport, hybrid logical clocks, vector clocks) order events *consistent with causality* without synchronized hardware clocks — but only vector clocks can *detect* concurrency.
- **Linearizable ID generators** are strictly stronger than logical clocks; they require a single-node timestamp oracle (TiDB/Percolator) or tightly synchronized clocks with uncertainty bounds (Spanner TrueTime).
- **Consensus** in all its forms — single-value agreement, CAS, shared logs (total-order broadcast), fetch-and-add, atomic commit — are mutually reducible; solving one solves them all.
- **Raft, (Multi-)Paxos, Zab, Viewstamped Replication** are the four mainstream consensus algorithms; all use epoch-numbered leaders and overlapping quorums so any two quorums intersect in at least one node.
- **Coordination services** (ZooKeeper, etcd, Consul) package consensus into a small cluster that exports locks, leases, fencing tokens, ephemeral sessions, and watches to thousands of application nodes.

---

## The Problem: Strong Consistency in Fault-Tolerant Replicated Systems

Replication gives us fault tolerance, but every replica is a potential source of staleness and conflict. Two philosophies coexist: **eventual consistency** makes replication visible to the app (multi-leader, leaderless), while **strong consistency** tries to pretend the system is a single node.

This chapter walks a tower of progressively stronger guarantees:

1. **Recency** — linearizability, "as-if-single-copy" behavior on a single object.
2. **Causal ordering** — logical clocks that label events so ordering respects happens-before.
3. **Total agreement** — consensus, where nodes agree on a single value (or log of values) despite crashes.

Each level is harder, more expensive, and — as FLP and Attiya-Welch tell us — bounded by fundamental limits.

---

## Linearizability: The Definition

A **linearizable register** behaves as if there is exactly one copy. For every read/write/CAS, there is a single moment between the client's request and response at which the operation took effect. Once one client reads a new value, no subsequent read (on any client) may return an older value.

```mermaid
sequenceDiagram
  participant A as Client A
  participant DB as Register x<br/>(initial = 0)
  participant B as Client B
  participant C as Client C
  C->>DB: write(x, 1)   [in flight]
  A->>DB: read(x)
  DB-->>A: 0             [pre-flip]
  Note over DB: atomic flip 0->1
  A->>DB: read(x)
  DB-->>A: 1             [post-flip]
  B->>DB: read(x)
  DB-->>B: 1             [must be 1, since A already saw 1]
  DB-->>C: ok            [write completes]
```

The key rule: once *any* client has observed the new value, *every* subsequent read must also see the new value (or newer). This is what distinguishes linearizability from weaker models like read-after-write or monotonic reads.

### Linearizability vs Serializability

| Aspect | Linearizability | Serializability |
|---|---|---|
| Scope | Single object (register) | Multiple objects inside a transaction |
| Guarantee type | Recency (real-time ordering) | Equivalent to *some* serial order |
| Stale reads allowed? | No | Yes (any serial order works) |
| Prevents write skew? | No (single-object only) | Yes (with strict serializability) |
| Usually implemented via | Consensus / single leader | 2PL, SSI, or serial execution |
| "Strict serializability" | = Serializable + Linearizable |  |

Many systems claim "strong consistency" but mean different things. ANSI SQL's "Serializable" says nothing about recency.

---

## Quantifying the Cost of Linearizability (Attiya-Welch)

Attiya and Welch proved a lower bound: in an asynchronous network where one-way delays vary between `d_min` and `d_max`, the response time of any linearizable read is at least `(d_max - d_min) / 2`, and every linearizable write takes at least `d_max - d_min`. You can't beat network-delay uncertainty.

In [ ]:
# Attiya-Welch style lower bound for a geo-replicated, linearizable store.
# We model one-way network delays with a min/max range and derive the
# UNAVOIDABLE extra latency any linearizable implementation must pay.

def attiya_welch(d_min_ms: float, d_max_ms: float) -> dict:
    """Return lower bounds on read/write response time for linearizability."""
    uncertainty = d_max_ms - d_min_ms
    return {
        "read_lower_bound_ms": uncertainty / 2,
        "write_lower_bound_ms": uncertainty,
    }

scenarios = [
    ("Same rack (LAN)",          0.05, 0.20),   # 50-200 microseconds
    ("Same DC, cross-rack",      0.30, 1.50),
    ("Cross-AZ (same region)",   0.80, 3.00),
    ("Cross-region (US east/west)", 30.0, 80.0),
    ("Cross-continent (US <-> EU)", 70.0, 200.0),
]

print(f"{'scenario':<32}{'d_min':>8}{'d_max':>8}{'read >=':>12}{'write >=':>12}")
print("-" * 72)
for name, lo, hi in scenarios:
    b = attiya_welch(lo, hi)
    print(f"{name:<32}{lo:>7.2f}ms{hi:>7.2f}ms"
          f"{b['read_lower_bound_ms']:>10.2f}ms{b['write_lower_bound_ms']:>10.2f}ms")

print()
print("Takeaway: linearizability's latency floor is set by NETWORK JITTER,")
print("not by raw RTT. A linearizable cross-continent write cannot respond in")
print("under ~130 ms, regardless of hardware, because the protocol must cover")
print("the worst-case message delay to preserve real-time ordering.")

---

## CAP, PACELC, and Majority Quorums

CAP says: if a partition (P) prevents some replicas from communicating, you must choose consistency (C) or availability (A). **PACELC** extends this: Else (no partition), trade Latency vs Consistency.

```mermaid
graph LR
  subgraph Region_A[US-East majority]
    A1[etcd-1]
    A2[etcd-2]
    A3[etcd-3]
  end
  subgraph Region_B[EU-West minority]
    B1[etcd-4]
    B2[etcd-5]
  end
  A1 -.replicate.-> B1
  A2 -.replicate.-> B2
  Client_EU -->|"write (must reach majority in US)"| B1
  Partition[["Network partition<br/>B side stops accepting writes<br/>(CP choice)"]]
```

### CAP vs PACELC at a glance

| Model | When partition occurs | When network is healthy |
|---|---|---|
| CAP (C-P) | Minority side unavailable; majority keeps serving | *not specified* |
| CAP (A-P) | All sides available but divergent | *not specified* |
| PACELC (PC/EC) | CP under partition | Consistency at cost of latency always |
| PACELC (PA/EL) | AP under partition | Low latency at cost of staleness |
| Examples PC/EC | etcd, ZooKeeper, Spanner, CockroachDB | |
| Examples PA/EL | Cassandra, DynamoDB (eventual), Riak | |

In [ ]:
# A simplified model of a 5-node etcd cluster spread across 2 regions.
# We compute the minimum write-commit latency for a linearizable write,
# which must reach a MAJORITY (3 of 5) including the leader.

import math

nodes = [
    ("us-east-1a", 0.5),  # ms from leader
    ("us-east-1b", 0.8),  # ms from leader
    ("us-east-1c", 1.1),  # ms from leader
    ("eu-west-1a", 70.0),
    ("eu-west-1b", 72.0),
]

leader = "us-east-1a"   # lowest-latency node elected leader
# A write must be acked by a quorum including the leader (implicit).
# So we need 2 additional acks out of the other 4.

quorum_size = math.ceil((len(nodes) + 1) / 2)  # 3 of 5
other_rtts_ms = sorted(n[1] for n in nodes if n[0] != leader)
needed_extra_acks = quorum_size - 1  # leader already has it

# RTT because leader proposes then waits for acks.
commit_latency_ms = 2 * other_rtts_ms[needed_extra_acks - 1]

print(f"Quorum size: {quorum_size} of {len(nodes)}")
print(f"Other nodes' one-way latencies (ms): {other_rtts_ms}")
print(f"Need {needed_extra_acks} extra acks; slowest required ack: {other_rtts_ms[needed_extra_acks - 1]} ms")
print(f"Minimum linearizable commit latency ~ {commit_latency_ms:.2f} ms")
print()

# Now flip the leader to an EU node — see how the cost changes:
leader_eu = "eu-west-1a"
other_rtts_ms_eu = sorted(
    abs(rtt - 70.0) if n.startswith("eu") else rtt + 70.0
    for n, rtt in nodes if n != leader_eu
)
commit_latency_eu_ms = 2 * other_rtts_ms_eu[needed_extra_acks - 1]
print(f"If leader is in EU, min commit latency ~ {commit_latency_eu_ms:.2f} ms")
print("Leader placement is a first-order latency decision for any CP system.")

---

## Logical Clocks: Lamport and Vector Clocks

Wall-clock time on different nodes disagrees by up to hundreds of milliseconds. **Logical clocks** give events timestamps whose order is consistent with the *happens-before* relation `->` without synchronized hardware.

```mermaid
sequenceDiagram
  participant A as Node A
  participant B as Node B
  participant C as Node C
  Note over A,C: Lamport rule:<br/>t_send = max(local, 0)+1<br/>t_recv = max(local, recv_ts)+1
  A->>A: tick -> (1,A)
  A->>B: msg "hi" ts=(1,A)
  B->>B: recv -> (2,B)
  B->>C: msg "fwd" ts=(3,B)
  C->>C: recv -> (4,C)
  C->>A: msg "ack" ts=(5,C)
  A->>A: recv -> (6,A)
```

### A minimal Lamport clock implementation

In [ ]:
# Lamport logical clocks + a 3-node chat simulation.
# Each node has a counter. On every local event it ticks.
# On receive it sets its counter to max(local, received) + 1.
# Tie-break by node id to get a TOTAL ORDER that respects causality.

from dataclasses import dataclass, field
from typing import List, Tuple

@dataclass(order=True)
class Stamp:
    counter: int
    node: str

@dataclass
class LamportNode:
    node_id: str
    counter: int = 0
    log: List[Tuple[Stamp, str]] = field(default_factory=list)

    def tick(self, event: str) -> Stamp:
        self.counter += 1
        s = Stamp(self.counter, self.node_id)
        self.log.append((s, f"local:{event}"))
        return s

    def send(self, event: str) -> Stamp:
        return self.tick(event)   # send is a local event too

    def recv(self, received: Stamp, event: str) -> Stamp:
        self.counter = max(self.counter, received.counter) + 1
        s = Stamp(self.counter, self.node_id)
        self.log.append((s, f"recv:{event} (from {received.node}@{received.counter})"))
        return s


# Simulate a 3-way chat. Ordering of deliveries mimics Figure 10-9:
# Aaliyah and Caleb post concurrently; Bryce sees Aaliyah first.
a = LamportNode("A")   # Aaliyah
b = LamportNode("B")   # Bryce
c = LamportNode("C")   # Caleb

s1 = a.send("Aaliyah: 'Anyone up for lunch?'")
s2 = c.send("Caleb:   'Tacos!'")                 # concurrent with s1
s3 = b.recv(s1, "<- Aaliyah")
s4 = b.recv(s2, "<- Caleb")
s5 = b.send("Bryce:   'Let's do tacos'")
s6 = a.recv(s5, "<- Bryce")
s7 = c.recv(s5, "<- Bryce")

all_events = sorted(
    [(stamp, node.node_id, desc)
     for node in (a, b, c) for stamp, desc in node.log]
)

print(f"{'ts':<8}{'node':<6}event")
print("-" * 70)
for stamp, node_id, desc in all_events:
    print(f"({stamp.counter},{stamp.node}){'':<2}{node_id:<6}{desc}")

# Causality sanity check: s1 -> s3 -> s5 -> s6
print()
print(f"s1={s1} -> s3={s3} -> s5={s5} -> s6={s6}")
print("All counters strictly increase along the causal chain. ✓")
print()
print("BUT: s1 and s2 both have counter=1; Lamport gives them a TOTAL ORDER")
print("via node-id tie-break, yet it CANNOT tell us they were concurrent.")
print("For that, we need vector clocks.")

### Lamport vs vector vs hybrid logical clocks

| Clock type | Size per stamp | Total order? | Detects concurrency? | Used by |
|---|---|---|---|---|
| Lamport | (counter, node_id) | Yes (with tie-break) | No | Academic, some request IDs |
| Hybrid Logical (HLC) | (phys_time, counter, node) | Yes | No | CockroachDB, YugabyteDB |
| Vector clock | one int per node | Partial order | **Yes** | Riak, Dynamo-style systems |
| Version vector | one int per replica | Partial order | Yes (per-key) | Riak, Voldemort |

---

## Linearizable ID Generators

Logical clocks are *causal*, not *linearizable*. If two operations never exchanged a message, Lamport cannot enforce real-time order between them. That gap leaks bugs.

```mermaid
sequenceDiagram
  participant Laptop as User A (laptop)
  participant Phone as User A (phone)
  participant Accounts as Accounts DB
  participant Photos as Photos DB
  participant Viewer as Snooping viewer
  Laptop->>Accounts: set privacy=private ts=(10, accts)
  Phone->>Photos: upload embarrassing photo ts=(7, photos)
  Note over Accounts,Photos: Photos DB's local counter<br/>was behind Accounts DB's.
  Viewer->>Accounts: read profile @ ts=8
  Accounts-->>Viewer: privacy=PUBLIC (ts=5 version)
  Viewer->>Photos: read photo @ ts=8
  Photos-->>Viewer: PHOTO (ts=7 version)
  Note over Viewer: Leak! Caused by<br/>non-linearizable IDs.
```

A **linearizable ID generator** guarantees: if op A completes before op B starts (real time), then ID(A) < ID(B), even with no message between them.

### Implementation strategies

| Strategy | How | Coordination | Region-friendly? | Example |
|---|---|---|---|---|
| Timestamp oracle | Single leader atomically increments and replicates a counter | One RTT to the oracle | No — oracle is single-region | TiDB/TiKV, Percolator |
| Spanner TrueTime | Read `TT.now()` range; commit-wait for `uncertainty` ms | None | Yes (needs GPS+atomic clocks) | Google Spanner |
| Hybrid logical clock | Physical time + Lamport counter | None | Yes, but *causal only* | CockroachDB, YugabyteDB |
| Lamport / vector | No real-time guarantee | None | Yes, but *causal only* | CRDT systems |

In [ ]:
# Spanner-style commit-wait overhead.
# TrueTime returns [earliest, latest]; uncertainty = latest - earliest.
# To guarantee external consistency, a transaction must wait AT LEAST
# `uncertainty` ms before revealing its commit timestamp.

def commit_wait_throughput(uncertainty_ms: float, keys: int = 1) -> dict:
    """Max serial commit throughput PER KEY when commit-waits are serialized."""
    if uncertainty_ms <= 0:
        return {"tps_per_key": float("inf"), "tps_total": float("inf")}
    tps_per_key = 1000.0 / uncertainty_ms   # one tx per uncertainty window
    return {"tps_per_key": tps_per_key, "tps_total": tps_per_key * keys}

print(f"{'uncertainty':<14}{'tps/key':>12}{'tps @ 1k keys':>16}")
print("-" * 50)
for u in [1.0, 4.0, 7.0, 10.0, 50.0]:
    r = commit_wait_throughput(u, keys=1000)
    print(f"{u:>7.1f} ms   {r['tps_per_key']:>10.1f}{r['tps_total']:>15.0f}")

print()
print("Spanner's published uncertainty is typically ~4-7 ms, bounding")
print("per-key write throughput to a few hundred tx/sec — but with thousands")
print("of keys, aggregate throughput scales. Shrinking `uncertainty` directly")
print("relaxes this ceiling, which is why TrueTime hardware matters.")

---

## Consensus and Its Equivalent Forms

Consensus asks: given N nodes that each *propose* a value, how do they *decide* on one value despite failures?

```mermaid
graph LR
  CAS[Compare-and-Set<br/>consensus #infin;]
  SVC[Single-value<br/>consensus]
  LOG[Shared Log /<br/>Total-Order Broadcast]
  AC[Atomic Commit]
  FA[Fetch-and-Add<br/>consensus 2]
  CAS <-->|equivalent| SVC
  SVC <-->|equivalent| LOG
  SVC <-->|equivalent| AC
  SVC -->|reduces to| FA
  FA -. fails when>2 proposers .-> SVC
  style FA fill:#fee
  style CAS fill:#efe
  style LOG fill:#efe
```

### The four formal properties

| Property | Meaning | Type |
|---|---|---|
| **Uniform agreement** | No two nodes decide differently | Safety |
| **Integrity** | A node decides at most one value (no flip-flop) | Safety |
| **Validity** | A decided value was proposed by *some* node | Safety |
| **Termination** | Every non-crashed node eventually decides | Liveness |

FLP: with crash failures and no timeouts/randomness, termination cannot be guaranteed. Practical systems sidestep FLP via **failure detectors** (timeouts) or randomization.

### Consensus numbers (Herlihy)

In [ ]:
# Consensus number: the MAX number of processes for which an object type
# can solve wait-free consensus. Higher = more powerful primitive.

primitives = [
    ("read/write register",           1,  "Trivial; cannot agree even for 2"),
    ("test-and-set",                  2,  "Enough for 2, no more"),
    ("fetch-and-add",                 2,  "Same — loser can't tell who won"),
    ("queue / stack",                 2,  "Same bound"),
    ("compare-and-set (CAS)",         float("inf"), "Universal primitive"),
    ("n-register assignment (n>=3)",  float("inf"), "Universal at size n"),
    ("total-order broadcast",         float("inf"), "Equivalent to consensus"),
    ("memory-to-memory move+swap",    float("inf"), "Universal"),
]

print(f"{'primitive':<34}{'consensus #':>14}   note")
print("-" * 78)
for name, cn, note in primitives:
    cn_s = "infinity" if cn == float("inf") else str(cn)
    print(f"{name:<34}{cn_s:>14}   {note}")

print()
print("PRACTICAL TAKEAWAY: any system built only on fetch-and-add or queues")
print("CANNOT solve consensus for >2 nodes. That's why real consensus boxes")
print("(Raft/Paxos) ultimately expose CAS or a log — both with consensus # inf.")

Deep implication: **solving one form solves them all.** A distributed CAS can implement a shared log, and vice versa. An atomic commit protocol can be built from consensus, and consensus can be built from atomic commit.

---

## Consensus Algorithms: Raft, Paxos, Zab, VR

All four use the same recipe:

1. Elect a **leader** in some *epoch* (a.k.a. term, ballot, view).
2. The leader proposes log entries; followers acknowledge if they haven't seen a higher epoch.
3. An entry commits once a **majority quorum** acks it.
4. On leader failure, start a new epoch; overlapping quorums guarantee the new leader has seen every committed entry.

```mermaid
sequenceDiagram
  participant F1 as Follower 1
  participant F2 as Follower 2
  participant L as Leader (epoch=7)
  participant F3 as Follower 3
  participant F4 as Follower 4
  Note over F1,F4: Phase 1: election (epoch 7), L collects votes from majority.
  F1->>L: vote(ok, lastLog=12)
  F2->>L: vote(ok, lastLog=12)
  F3->>L: vote(ok, lastLog=11)
  Note over L: Has majority 3 of 5, leader wins epoch 7.
  Note over F1,F4: Phase 2: log replication
  L->>F1: append(entry=42, epoch=7, prev=12)
  L->>F2: append(entry=42, epoch=7, prev=12)
  L->>F3: append(entry=42, epoch=7, prev=12)
  L->>F4: append(entry=42, epoch=7, prev=12)
  F1-->>L: ok
  F2-->>L: ok
  F3-->>L: ok
  Note over L: Majority acked, entry 42 is COMMITTED and followers informed.
```

### Raft vs Multi-Paxos vs Zab vs VR

| Dimension | Raft | Multi-Paxos | Zab | Viewstamped Replication |
|---|---|---|---|---|
| Epoch name | term | ballot number | epoch/zxid | view |
| Election | Randomized timeout + vote request | Prepare/promise phase | Leader election module (FLE) | View-change protocol |
| Log model | Append-only, strong leader | Per-slot consensus, gaps allowed | Primary-ordered broadcast (FIFO) | Append-only, primary-based |
| Leader catch-up rule | New leader must have longest up-to-date log | New leader re-proposes highest-numbered pending | Leader with highest zxid | Primary synced via DO-VIEW-CHANGE |
| Reigning impl | etcd, Consul, CockroachDB, TiKV | Google Chubby, Spanner | Apache ZooKeeper | Obsolete direct use; basis for others |
| Formal proof | TLA+ spec, easier teaching | Harder (many variants) | Paper + ZK use in prod | Original 1988 paper |

### Quorum overlap — the mathematical heartbeat of consensus

In [ ]:
# Any two quorums of size q in an N-node cluster must intersect in >= 1 node.
# That's the whole safety argument: a new leader's election quorum overlaps
# the previous leader's commit quorum, so the new leader MUST have seen
# every committed entry.

def quorum_intersect_size(N: int, q: int) -> int:
    """Minimum possible intersection of two quorums of size q in N nodes."""
    # By inclusion-exclusion: |A ∩ B| >= |A| + |B| - N = 2q - N
    return max(0, 2 * q - N)

def min_quorum_for_majority(N: int) -> int:
    return N // 2 + 1

print(f"{'N':>4}{'majority q':>12}{'2q - N':>10}{'tolerates':>12}")
print("-" * 44)
for N in [1, 3, 5, 7, 9, 11]:
    q = min_quorum_for_majority(N)
    ovl = quorum_intersect_size(N, q)
    tolerated = N - q
    print(f"{N:>4}{q:>12}{ovl:>10}{tolerated:>12}")

print()
print("Rule: 2q > N is the ONLY requirement for strict majority quorums.")
print("With 2q > N, any election quorum and any commit quorum share at")
print("least one node, so the new leader inherits every committed entry.")
print("N=2f+1 tolerates f failures with the smallest cluster.")
print()

# Flexible Paxos: you can trade phase-1 vs phase-2 quorum sizes
# as long as ELECTION and REPLICATION quorums intersect.
def flexible_paxos_ok(N: int, q1: int, q2: int) -> bool:
    return q1 + q2 > N

print("Flexible Paxos example:")
for (q1, q2) in [(3, 3), (4, 2), (2, 4), (2, 2)]:
    ok = flexible_paxos_ok(5, q1, q2)
    print(f"  N=5, election q1={q1}, replication q2={q2} -> safe? {ok}")
print("Smaller replication quorums give faster commits in the common case.")

---

## Coordination Services: ZooKeeper, etcd, Consul

Running Raft/Paxos is complex. Rather than embed it in every application, we outsource coordination to a **small, specialized cluster** (usually 3-5 nodes) that exports a simple key-value API plus coordination primitives.

```mermaid
graph TB
  subgraph Coordination_Cluster["Coordination cluster (consensus, 3-5 nodes)"]
    Z1[etcd-1 leader]
    Z2[etcd-2]
    Z3[etcd-3]
    Z1 <-->|Raft| Z2
    Z1 <-->|Raft| Z3
    Z2 <-->|Raft| Z3
  end
  subgraph App_Tier["Application tier (many thousands of nodes)"]
    A1[app-1]
    A2[app-2]
    A3[app-3]
    A4[...]
    An[app-N]
  end
  A1 -->|"lock, lease, watch"| Z1
  A2 -->|"read config"| Z1
  A3 -->|"register ephemeral"| Z1
  An -->|"heartbeat session"| Z1
```

### Features and their uses

| Feature | What it does | Typical use |
|---|---|---|
| Linearizable CAS on keys | Atomic key updates | Leader election, distributed locks |
| **Ephemeral nodes** | Auto-delete on session loss | Membership, failure detection |
| **Leases / TTL** | Grant a key for bounded time | Leader leases, job tokens |
| **Fencing tokens** (monotonic `zxid`/`revision`) | Strictly increasing IDs per update | Avoid split-brain writes to shared storage |
| **Watches** | Change notifications | Service discovery, dynamic config, sharding |
| Total-order broadcast | Ordered log of changes | State machine replication by clients |

### When NOT to reach for ZooKeeper/etcd

- **Service discovery** where stale answers are acceptable — a simple eventually-consistent DNS or gossip is often enough.
- **Per-request coordination** in the hot path — the coordination cluster's throughput (thousands of writes/sec) becomes a bottleneck.
- **Large data sets** — designed for small, metadata-scale state (MBs, not GBs).

---

## Trade-offs and Alternatives

| Dimension | Strong consistency (coordination-based) | Eventual consistency (coordination-avoidance) |
|---|---|---|
| Correctness reasoning | "Single-node illusion" | App must handle conflicts, merges |
| Latency floor | `d_max - d_min` (Attiya-Welch) | Local-region response possible |
| Availability under partition | Minority side unavailable | All sides available, divergent |
| Common implementations | Single-leader, Raft/Paxos, Spanner, 2PC | Multi-leader, leaderless, CRDTs |
| Scale ceiling | Bounded by leader/quorum throughput | Scales horizontally with replicas |
| Use when... | Invariants across objects, correctness > latency | Offline-first apps, massive scale, local-first (Ch 13) |

Chapter 13 will pick up where this leaves off: can we get *most* of consistency's correctness without paying consensus's coordination tax?

---

## Key Takeaways

1. **Linearizability = recency guarantee on a single object.** Don't confuse it with serializability (multi-object, transactional, no real-time guarantee).
2. **Every linearizable system pays for network uncertainty.** Attiya-Welch: minimum read time is `(d_max - d_min)/2`, write time is `d_max - d_min`. Geo-distribution amplifies this dramatically.
3. **Logical clocks give causal order, not real-time order.** Lamport and HLC total-order events consistent with happens-before; only vector clocks can *detect* concurrency.
4. **Linearizable IDs are strictly harder than causal IDs.** Either concentrate on one node (timestamp oracle, TiDB) or use synchronized-clock bounds (Spanner TrueTime) — you cannot shard them.
5. **Consensus equals CAS equals total-order broadcast equals atomic commit.** Solving one solves all. Fetch-and-add is weaker (consensus number 2).
6. **All practical algorithms share the same shape.** Epoch-numbered leader + majority quorum for both election and replication. Safety comes from `2q > N` overlap; liveness from timeouts.
7. **Centralize consensus in a coordination service.** Don't build Raft into every app. ZooKeeper/etcd/Consul are how you get locks, leader election, fencing, and membership without bespoke distributed protocols.

---

## See Also

- [[01-linearizability]]
- [[02-cap-theorem-and-cost-of-linearizability]]
- [[03-logical-clocks]]
- [[04-linearizable-id-generators]]
- [[05-consensus-and-its-equivalent-forms]]
- [[06-consensus-algorithms]]
- [[07-coordination-services]]